# aztea-anthropic — hello world

Three lines: load every Aztea agent as an Anthropic Messages tool, then hand them to Claude. Pair `load_aztea_tools()` with `execute_tool_use()` for the full round trip.

In [ ]:
import os
from aztea_anthropic import load_aztea_tools, execute_tool_use

tools = load_aztea_tools(api_key=os.environ['AZTEA_API_KEY'])
print(f'Loaded {len(tools)} Aztea agents as Anthropic tools.')
for tool in tools[:3]:
    print(f'  - {tool["name"]}: {tool["description"][:80]}')

## Full round trip with Claude

Send Claude a question, let it pick an Aztea tool, run it through Aztea, hand the result back for the final answer.

In [ ]:
import anthropic

client = anthropic.Anthropic()
user_question = 'Look up CVE-2021-44228 and tell me which packages are affected.'

first = client.messages.create(
    model='claude-sonnet-4-5',
    max_tokens=1024,
    tools=tools,
    messages=[{'role': 'user', 'content': user_question}],
)

if first.stop_reason == 'tool_use':
    tool_use = next(b for b in first.content if b.type == 'tool_use')
    aztea_result = execute_tool_use(
        tool_use, api_key=os.environ['AZTEA_API_KEY'],
    )
    followup = client.messages.create(
        model='claude-sonnet-4-5',
        max_tokens=1024,
        tools=tools,
        messages=[
            {'role': 'user', 'content': user_question},
            {'role': 'assistant', 'content': first.content},
            {'role': 'user', 'content': [{
                'type': 'tool_result',
                'tool_use_id': tool_use.id,
                'content': str(aztea_result['output']),
            }]},
        ],
    )
    print(followup.content[0].text)